In [1]:
from timescale import TimescaleDBManager
from dotenv import load_dotenv
from sqlalchemy import create_engine
import pandas as pd
import os
import matplotlib.pyplot as plt
import numpy as np
import joblib
import json
import re##

#from sklearn.model_selec#tion import train_test_split, GridSearchCV
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay, accuracy_score, 
    precision_score, recall_score, f1_score, classification_report,
    roc_auc_score, roc_curve, RocCurveDisplay
)
#from sklearn.inspection import permutation_importance
from sklearn.preprocessing import LabelEncoder, StandardScaler

# from tensorflow.keras import layers, models
# from tensorflow.keras.models import Model, Sequential, load_model
# from tensorflow.keras.layers import Input, Dense, BatchNormalization, Dropout
# from tensorflow.keras.optimizers import Adam
# from tensorflow.keras.callbacks import Callback, EarlyStopping, ReduceLROnPlateau

# from scikeras.wrappers import KerasRegressor
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier


# import optuna
# from optuna.samplers import TPESampler
# import tensorflow as tf
# import seaborn as sns
# from functools import partial
# from tqdm import tqdm
import gc
import sys
import warnings


from sklearn.ensemble import VotingClassifier


In [2]:
load_dotenv()
user_env        = os.getenv("USER_DB")
pass_env        = os.getenv("PASS_DB")
host_env        = os.getenv("HOST_DB", "localhost")
port_env        = os.getenv("DB_PORT_EXPOSED", "5432")
db_env          = os.getenv("NAME_DB", "tfm_db")
FOLDER_SALIDA   = os.getenv("FOLDER_SALIDA")
FOLDER_TMP      = os.getenv("FOLDER_TMP")

SEED            = 100#int(os.getenv("SEED"))

db = TimescaleDBManager(
    user=user_env,
    password=pass_env,
    host=host_env,
    port=port_env,
    dbname=db_env
)
db.conectar()
conexion_postgresql = db.connection

#esto es para usar una conexion postgrews para pandas que da un warning indicando que solo esta testeado para sqlalchemy
engine = create_engine(f'postgresql+psycopg2://{user_env}:{pass_env}@{host_env}:{port_env}/{db_env}')

[OK] Conexión establecida con TimescaleDB.


In [3]:
tabla_origen = 'vectores'
tabla_split  = 'vectores_split'
filename_lista_metric_name_vector = f'{FOLDER_SALIDA}/lista_metric_name_vector.json'

In [4]:
SQL=f'select * from {tabla_origen}'
df = pd.read_sql(SQL, engine)


In [5]:
lista_label = df['label'].unique().tolist()

In [6]:
len(lista_label)

54

In [7]:
df_result=df[ ['vector_id','execution_name','label'] ].copy()


In [8]:
df_result

,vector_id,execution_name,label
0,190754,light-oauth2-data-1719592986.tar,correct
1,190755,light-oauth2-data-1719592986.tar,correct
2,190756,light-oauth2-data-1719592986.tar,correct
3,190757,light-oauth2-data-1719592986.tar,correct
4,190843,light-oauth2-data-1719594842.tar,invalid_code_verifier_format_PKCE_400
...,...,...,...
90185,280939,light-oauth2-data-1725866154.tar,delete_user_404_no_user
90186,280940,light-oauth2-data-1725866154.tar,update_password_401_wrong_password
90187,280941,light-oauth2-data-1725866154.tar,update_password_404_user_not_found
90188,280942,light-oauth2-data-1725866154.tar,update_password_400_not_match


In [9]:
with open(filename_lista_metric_name_vector, "r", encoding="utf-8") as f:
    lista_metric_name_vector = json.load(f)

In [48]:
def tabla_comparacion_modelos(lista_filenames):
    for filename in lista_filenames:
        paquete_modelo = joblib.load(filename)
        model_name      = paquete_modelo['model_name']
        accuracy        = paquete_modelo['resultado']['accuracy']
        macro_f1        = paquete_modelo['resultado']['precision']
        macro_precision = paquete_modelo['resultado']['recall']
        macro_recall    = paquete_modelo['resultado']['f1']
        del paquete_modelo
        print(f"Model = {model_name:70s} Accuracy = {accuracy:.4f} | Macro F1 = {macro_f1:.4f} | Macro Precision = {macro_precision:.4f} | Macro Recall = {macro_recall:.4f}")
        gc.collect()
        

In [12]:
def reporte_uso_memoria():

    # Lista para almacenar la info y poder ordenarla
    variables_info = []
    
    for nombre, obj in list(globals().items()):
        # Evitamos variables del sistema que empiezan por '_'
        if not nombre.startswith('_'):
            peso_mb = sys.getsizeof(obj) / (1024 * 1024)
            
            # Cálculo más preciso para DataFrames de pandas o matrices de numpy
            if hasattr(obj, 'memory_usage'):
                try:
                    peso_mb = obj.memory_usage(deep=True).sum() / (1024 * 1024)
                except:
                    pass
            elif hasattr(obj, 'nbytes'):
                peso_mb = obj.nbytes / (1024 * 1024)
                
            variables_info.append({
                'nombre': nombre,
                'tipo': type(obj).__name__,
                'peso': peso_mb
            })
    
    # Ordenar de mayor a menor peso (descendente)
    variables_info = sorted(variables_info, key=lambda x: x['peso'], reverse=True)
    
    # Imprimir la tabla formateada
    print(f"{'Variable':<30} | {'Tipo':<20} | {'Tamaño (MB)':<12}")
    print("-" * 68)
    for var in variables_info:
        print(f"{var['nombre']:<30} | {var['tipo']:<20} | {var['peso']:>10.2f} MB")

In [13]:


def X_matrix_sel_data(X_matrix,lista_feature_names,lista_idx_filas=None,lista_feature_sel=None):
    #X_matrix es la matriz completa
    #lista_feature_names son los nombres de las columnas
    #lista_idx_filas es la lista de indices de las filas que tomaremos
    #lista_feature_sel es la lista de nombres de las columnas que tomaremos, es un subconjunto de lista_feature_names
    if lista_idx_filas is None:
        lista_idx_filas   = range(0,X_matrix.shape[0])
    if lista_feature_sel is None:
        lista_feature_sel = lista_feature_names.copy()

    lista_feature_idx = []
    idx=0
    for feature in lista_feature_names:
        if feature in lista_feature_sel:
            lista_feature_idx.append(idx)
        idx += 1        

    # 1. Filtramos primero las filas
    X_temp = X_matrix[list(lista_idx_filas), :]
    
    # 2. Filtramos después las columnas sobre el resultado anterior
    X_matrix_sub = X_temp[:, lista_feature_idx]
    
    return X_matrix_sub

    


In [14]:
def evaluar_modelo(paquete_modelo,lista_feature_names,X_test,Y_test_etiquetas,encoder_etiquetas):
    #lista_feature_names  es el nombre de las columnas de X_test
    lista_feature_names_modelo = paquete_modelo['lista_features']
    model = paquete_modelo['model']
    lista_feature_idx = []
    idx=0
    for feature in lista_feature_names:
        if feature in lista_feature_names_modelo:
            lista_feature_idx.append(idx )
        idx += 1
        
    X_test_sub                                  = X_test[:, lista_feature_idx]
    Y_pred                                      = model.predict(X_test_sub)
    Y_pred_tipos                                = encoder_etiquetas.inverse_transform(Y_pred)
    try:
        Y_pred_probabilidades                       = model.predict_proba(X_test_sub)
    except:
        Y_pred_probabilidades                       = None
        
    reporte_dict                                = classification_report(Y_test_etiquetas, Y_pred_tipos, output_dict=True, zero_division=0)
    classification_report_str                   = classification_report(Y_test_etiquetas, Y_pred_tipos)    
    accuracy                                    = reporte_dict['accuracy']
    macro_f1                                    = reporte_dict['macro avg']['f1-score']
    macro_precision                             = reporte_dict['macro avg']['precision']
    macro_recall                                = reporte_dict['macro avg']['recall']

    resultado                                   = {'accuracy': accuracy,  'precision':macro_precision   , 'recall':macro_recall  , 'f1':macro_f1 }

    paquete_modelo['resultado']                 = resultado
    paquete_modelo['probabilidades']            = Y_pred_probabilidades
    paquete_modelo['reporte_dict']              = reporte_dict
    paquete_modelo['classification_report_str'] = classification_report_str
    
    return resultado,reporte_dict,Y_pred_probabilidades

def predict_modelo(paquete_modelo,lista_feature_names,X_test):
    lista_feature_names_modelo = paquete_modelo['lista_features']
    model = paquete_modelo['model']
    lista_feature_idx = []
    idx=0
    for feature in lista_feature_names:
        if feature in lista_feature_names_modelo:
            lista_feature_idx.append(idx )
        idx += 1

    X_test_sub                                  = X_test[:, lista_feature_idx]        
    Y_pred                                      = model.predict(X_test_sub)   
    return Y_pred

    

def get_lista_feature_sin_lag(lista_features):
    lista_features_result=[]
    for feature in lista_features:
        feature_limpia = re.sub(r'\(lag_\d+\)$', '', feature)
        if feature_limpia not in  lista_features_result:
            lista_features_result.append(feature_limpia)
    return lista_features_result

    

In [15]:
# #para optimizacion con OPTUNA
# def print_callback(study, trial):
#     # Si el trial falló por cualquier motivo, lo saltamos visualmente
#     if trial.state != optuna.trial.TrialState.COMPLETE:
#         pbar.update(1)
#         return

#     current_acc = trial.user_attrs.get('accuracy', trial.value)
#     current_params = trial.params
    
#     # Línea 1: Información de la iteración actual
#     print(f"Trial {trial.number:2d}                | Actual Acc: {current_acc:.4f} | Parámetros: {current_params}")
    
#     # Línea 2: Mejor resultado histórico hasta ahora
#     print(f"   -> MEJOR HASTA AHORA | Mejor Acc:  {study.best_value:.4f} | Parámetros: {study.best_params}")
#     print("-" * 100)
    
#     # Actualización de la barra de progreso
#     pbar.update(1)
#     pbar.set_postfix({
#         "Mejor_Acc": f"{study.best_value:.4f}",
#         "Trial": trial.number
#     })


In [16]:
# def objective_universal(trial, config, clasificador_cls, X_train, X_test, Y_train_encoded, Y_test_encoded):
#     params = {}
    
#     for key, values in config.items():
#         params[key] = trial.suggest_categorical(key, values)

#     params['random_state'] = SEED
#     if 'n_jobs' in clasificador_cls().get_params():
#         params['n_jobs'] = -1

#     model = clasificador_cls(**params)
#     model.fit(X_train, Y_train_encoded)
    
#     preds = model.predict(X_test)
#     accuracy = accuracy_score(Y_test_encoded, preds)
    
#     trial.set_user_attr('accuracy', accuracy)
#     return accuracy

In [17]:
# def obtener_importancias_agrupadas(importancias, lista_features):
#     """
#     Procesa las importancias de un modelo, elimina los sufijos de lag/sum,
#     las agrupa por característica base y calcula cuántas características tienen importancia cero.
#     """
#     # 1. Crear el DataFrame inicial con las características y sus importancias
#     df_importancias = pd.DataFrame({
#         'caracteristica_con_lag': lista_features,
#         'importancia': importancias
#     }).sort_values(by='importancia', ascending=False).reset_index(drop=True)
    
#     # 2. Limpiar los sufijos de lags y sumas para obtener la característica base
#     df_importancias['caracteristica'] = (
#         df_importancias['caracteristica_con_lag']
#         .str.replace(r'\(lag_\d+\)', '', regex=True)
#         .str.replace(r'\(sum\)', '', regex=False)
#     )
    
#     # 3. Agrupar por característica base sumando sus importancias
#     df_importancias_agrupada = (
#         df_importancias.groupby('caracteristica', as_index=False)['importancia']
#         .sum()
#         .sort_values(by='importancia', ascending=False)
#         .reset_index(drop=True)
#     )
    
#     # 4. Contar cuántas características agrupadas tienen una importancia igual a 0
#     caracteristicas_cero = df_importancias_agrupada[df_importancias_agrupada['importancia'] == 0]
    
#    return df_importancias_agrupada, importancias, len(caracteristicas_cero)

# <span style="color:#2ca02c">1. Recuperamos separación ya hecha en conjuntos de train, val y test</span>

In [18]:
SQL=f'select * from {tabla_split}'
df_split = pd.read_sql(SQL, engine)

In [19]:
df_split

,vector_id,split_type
0,190754,test
1,190755,train
2,190756,train
3,190757,test
4,190843,train
...,...,...
90185,280939,train
90186,280940,val
90187,280941,train
90188,280942,val


In [20]:
df_result = pd.merge(df_result, df_split, on="vector_id", how="inner")

In [21]:
df_result

,vector_id,execution_name,label,split_type
0,190754,light-oauth2-data-1719592986.tar,correct,test
1,190755,light-oauth2-data-1719592986.tar,correct,train
2,190756,light-oauth2-data-1719592986.tar,correct,train
3,190757,light-oauth2-data-1719592986.tar,correct,test
4,190843,light-oauth2-data-1719594842.tar,invalid_code_verifier_format_PKCE_400,train
...,...,...,...,...
90185,280939,light-oauth2-data-1725866154.tar,delete_user_404_no_user,train
90186,280940,light-oauth2-data-1725866154.tar,update_password_401_wrong_password,val
90187,280941,light-oauth2-data-1725866154.tar,update_password_404_user_not_found,train
90188,280942,light-oauth2-data-1725866154.tar,update_password_400_not_match,val


In [22]:
dic_idx_train = {}
#dic_idx_resto = {}
dic_idx_val   = {}
dic_idx_test  = {}

In [23]:
for label in lista_label:
    idx_train = df_result[  (df_result['label']==label) & (df_result['split_type'] == 'train') ].index.values
    idx_val   = df_result[  (df_result['label']==label) & (df_result['split_type'] == 'val'  ) ].index.values
    idx_test  = df_result[  (df_result['label']==label) & (df_result['split_type'] == 'test') ].index.values
    dic_idx_train[label] = idx_train
    dic_idx_val[label]   = idx_val
    dic_idx_test[label]  = idx_test

<span style="color:#2ca02c"> Convetimos los vectores del df en una matriz, seran los datos que pasemos a los algoritmo   </span>


In [24]:
lista_matrix        = []
lista_feature_names = []

X_matrix1 = np.array(df['vector'].tolist())
X_matrix1.shape


(90190, 354)

In [25]:
N_puntos=3
lista_feature = []
#lista_feature_orig = lista_metric_name_vector.copy()
# for lag in range(0,N_puntos):
#     for feature in lista_feature_orig:
#         lista_feature.append(f'{feature}(lag_{lag})')
lista_feature=lista_metric_name_vector.copy()
lista_feature_names1 = lista_feature.copy()
lista_feature_names.extend(lista_feature_names1)
lista_matrix.append(X_matrix1)






In [ ]:
lista_feature_names

In [27]:
lista_features_names_3puntos = lista_feature.copy()

In [28]:
num_columnas_total = X_matrix1.shape[1]
tercio_n = num_columnas_total // N_puntos

# Definimos los límites de los 3 tercios
inicio_1, fin_1 = 0, tercio_n
inicio_2, fin_2 = tercio_n, tercio_n * 2
inicio_3, fin_3 = tercio_n * 2, num_columnas_total  # Coge el resto hasta el final por si acaso

# 2. Extraemos cada tercio como una submatriz
t1 = X_matrix1[:, inicio_1:fin_1]
t2 = X_matrix1[:, inicio_2:fin_2]
t3 = X_matrix1[:, inicio_3:fin_3]

# 3. Sumamos las columnas homólogas de los tres tercios (columna i del tercio 1 + columna i del tercio 2 + columna i del tercio 3)
# Como t1, t2 y t3 tienen exactamente la misma forma (n_filas, tercio_n), la suma de NumPy los alinea automáticamente.
X_matrix1_sum = t1 + t2 + t3
del t1,t2,t3
gc.collect()

0

In [29]:
lista_feature_names1_sum = []
for feature in lista_feature_names1:
    feature_sum = re.sub(r'\(lag_\d+\)', '(sum)', feature)
    if feature_sum not in lista_feature_names1_sum:
        lista_feature_names1_sum.append(feature_sum)


In [30]:
lista_matrix.append(X_matrix1_sum)
lista_feature_names.extend(lista_feature_names1_sum)

In [31]:
X_matrix = np.concatenate(lista_matrix, axis=1)

In [32]:
print(f'len(lista_feature_names1_sum)  = {len(lista_feature_names1_sum)}')
print(f'len(lista_feature_names1)      = {len(lista_feature_names1)}')
print(f'len(lista_feature_names)       = {len(lista_feature_names)}')

len(lista_feature_names1_sum)  = 118
len(lista_feature_names1)      = 354
len(lista_feature_names)       = 472


In [33]:
lista_feature_names_1punto_sum=lista_feature_names1_sum.copy()

In [34]:
len(lista_feature_names_1punto_sum)

118

# <span style="color:#2ca02c">2. Preparacion de datos para entrenamiento/test</span>

In [35]:
#1. Establecemos la semilla para tener reproducibilidad
rng = np.random.default_rng(SEED)

# 2. Primero calculamos los mínimos por clase asegurando barajado aleatorio previo si se desea
min_train = float('inf')
min_test  = float('inf')
min_val   = float('inf')

for label in lista_label:
    if label == 'correct':
        continue
    
    min_train = min(min_train, len(dic_idx_train[label]))
    min_test  = min(min_test,  len(dic_idx_test[label]))
    min_val   = min(min_val,   len(dic_idx_val[label]))

# 3. Inicializamos los arrays de índices vacíos
idx_train = np.array([], dtype=int)
idx_test  = np.array([], dtype=int)
idx_val   = np.array([], dtype=int)

# 4. Bucle de selección aleatoria balanceada usando la SEED
for label in lista_label:
    if label == 'correct':
        continue
    
    # Barajamos los índices de forma aleatoria con la semilla antes de recortar
    arr_train = rng.permutation(dic_idx_train[label])
    arr_test  = rng.permutation(dic_idx_test[label])
    arr_val   = rng.permutation(dic_idx_val[label])
    
    # Concatenamos la cantidad mínima común de forma aleatoria, todas las clases tienen la misma cantidad de datos para que esten balanceadas
    idx_train = np.concatenate((idx_train, arr_train[:min_train]))
    idx_test  = np.concatenate((idx_test,  arr_test[:min_test]))
    idx_val   = np.concatenate((idx_val,   arr_val[:min_val]))

In [36]:
X_train = X_matrix_sel_data(X_matrix,lista_feature_names,lista_idx_filas=idx_train,lista_feature_sel=lista_feature_names1)
print(X_train.shape)
X_test  = X_matrix_sel_data(X_matrix,lista_feature_names,lista_idx_filas=idx_test ,lista_feature_sel=lista_feature_names1)
print(X_test.shape)
X_val   = X_matrix_sel_data(X_matrix,lista_feature_names,lista_idx_filas=idx_test ,lista_feature_sel=lista_feature_names1)
print(X_val.shape)

(6572, 354)
(2226, 354)
(2226, 354)


In [37]:
X_train_sum = X_matrix_sel_data(X_matrix,lista_feature_names,lista_idx_filas=idx_train,lista_feature_sel=lista_feature_names1_sum)
print(X_train_sum.shape)
X_test_sum  = X_matrix_sel_data(X_matrix,lista_feature_names,lista_idx_filas=idx_test ,lista_feature_sel=lista_feature_names1_sum)
print(X_test_sum.shape)
X_val_sum   = X_matrix_sel_data(X_matrix,lista_feature_names,lista_idx_filas=idx_test ,lista_feature_sel=lista_feature_names1_sum)
print(X_val_sum.shape)

(6572, 118)
(2226, 118)
(2226, 118)


In [38]:
X_matrix.shape

(90190, 472)

In [39]:
#esta parte comentada hay que hacerla si no hemos crado antes el label encoder, si lo hemos creado , recuperamos el label encoder de fichero
# encoder_etiquetas = LabelEncoder()

ruta_encoder = f'{FOLDER_SALIDA}/models/encoder_etiquetas.joblib'
encoder_etiquetas = joblib.load(ruta_encoder)


for indice, etiqueta in enumerate(encoder_etiquetas.classes_):
    print(f"Índice {indice} --> {etiqueta}")

Índice 0 --> access_token_auth_header_error_401
Índice 1 --> access_token_authorization_form_401
Índice 2 --> access_token_client_id_not_found_404
Índice 3 --> access_token_client_secret_wrong_401
Índice 4 --> access_token_form_urlencoded_400
Índice 5 --> access_token_illegal_grant_type_400
Índice 6 --> access_token_missing_authorization_header_400
Índice 7 --> authorization_code_client_id_missing_400
Índice 8 --> authorization_code_invalid_client_id_404
Índice 9 --> authorization_code_invalid_password_401
Índice 10 --> authorization_code_missing_response_type_400
Índice 11 --> authorization_code_response_not_code_400
Índice 12 --> code_challenge_invalid_format_pkce_400
Índice 13 --> code_challenge_too_long_pkce_400
Índice 14 --> code_challenge_too_short_pkce_400
Índice 15 --> code_verifier_missing_pkce_400
Índice 16 --> code_verifier_too_long_pkce_400
Índice 17 --> code_verifier_too_short_pkce_400
Índice 18 --> delete_client_404_no_client
Índice 19 --> delete_service_404_no_service
Ín

In [40]:
Y_train_etiquetas = df.loc[idx_train , 'label']
Y_test_etiquetas  = df.loc[idx_test  , 'label']
Y_val_etiquetas   = df.loc[idx_val  , 'label']

Y_train_encoded = encoder_etiquetas.transform(Y_train_etiquetas)
Y_test_encoded  = encoder_etiquetas.transform(Y_test_etiquetas)
Y_val_encoded   = encoder_etiquetas.transform(Y_val_etiquetas)

In [41]:
Y_train_encoded

array([31, 31, 31, ..., 43, 43, 43], shape=(6572,))

In [42]:
reporte_uso_memoria()

Variable                       | Tipo                 | Tamaño (MB) 
--------------------------------------------------------------------
X_matrix                       | ndarray              |     324.78 MB
df                             | DataFrame            |     296.94 MB
X_matrix1                      | ndarray              |     243.59 MB
X_matrix1_sum                  | ndarray              |      81.20 MB
df_result                      | DataFrame            |      20.79 MB
X_train                        | ndarray              |      17.75 MB
X_test                         | ndarray              |       6.01 MB
X_val                          | ndarray              |       6.01 MB
df_split                       | DataFrame            |       5.97 MB
X_train_sum                    | ndarray              |       5.92 MB
X_test_sum                     | ndarray              |       2.00 MB
X_val_sum                      | ndarray              |       2.00 MB
Y_train_etiquetas     

In [43]:
del df,X_matrix1,X_matrix1_sum
gc.collect()

20

# <span style="color:#2ca02c">3. Listado y comparacion de modelos</span>

In [77]:
lista_filenames=[
    'result/models/RandomForest (metricas 3 puntos)(todas las features).joblib',
    'result/models/RandomForest (metricas 3 puntos)   N_feat=19.joblib',
    'result/models/RandomForest (metricas 1 punto sum)(todas las features).joblib',
    'result/models/RandomForest (metricas 1 punto sum) N_feat=29.joblib',
    
    'result/models/XGBoost (metricas 3 puntos)(todas las features).joblib',
    'result/models/XGBoost (metricas 3 puntos)   N_feat=31.joblib',
    'result/models/XGBoost (metricas 1 punto sum)(todas las features).joblib',
    'result/models/XGBoost (metricas 1 punto sum) N_feat=41.joblib',
    
    'result/models/LightGBM (metricas 3 puntos)(todas las features).joblib',
    'result/models/LightGBM (metricas 3 puntos) N_feat=78.joblib',
    'result/models/LightGBM (metricas 1 punto sum)(todas las features).joblib',
    'result/models/LightGBM (metricas 1 punto sum) N_feat=36.joblib',
]

In [78]:
lista_modelos=[]
for filename in lista_filenames:
    paquete_modelo = joblib.load(filename)
    lista_modelos.append(paquete_modelo)



In [79]:
print('Comparacion de modelos originales obtenidos con metricas_logs')
print('-'*80)
tabla_comparacion_modelos(lista_filenames)

Comparacion de modelos originales obtenidos con metricas_logs
--------------------------------------------------------------------------------
Model = RandomForest (metricas 3 puntos)(todas las features)                   Accuracy = 0.7451 | Macro F1 = 0.7536 | Macro Precision = 0.7501 | Macro Recall = 0.7505
Model = RandomForest (metricas 3 puntos)   N_feat=19                           Accuracy = 0.7615 | Macro F1 = 0.7689 | Macro Precision = 0.7656 | Macro Recall = 0.7662
Model = RandomForest (metricas 1 punto sum)(todas las features)                Accuracy = 0.7004 | Macro F1 = 0.7094 | Macro Precision = 0.7059 | Macro Recall = 0.7062
Model = RandomForest (metricas 1 punto sum) N_feat=29                          Accuracy = 0.7134 | Macro F1 = 0.7214 | Macro Precision = 0.7187 | Macro Recall = 0.7187
Model = XGBoost (metricas 3 puntos)(todas las features)                        Accuracy = 0.7695 | Macro F1 = 0.7763 | Macro Precision = 0.7721 | Macro Recall = 0.7735
Model = XGBoost (

In [47]:
gc.collect()

0

# <span style="color:#2ca02c">4. Ensemble de modelos</span>

## <span style="color:#2ca02c">4.1. Ensemble de modelos de 3 puntos con todas las features</span>

In [53]:

#hacemos esto para preparar los datos en las mismas variables todos los algoritmos

lista_features = lista_features_names_3puntos

X_train = X_matrix_sel_data(X_matrix,lista_feature_names,lista_idx_filas=idx_train,lista_feature_sel=lista_features)
print(X_train.shape)
X_test  = X_matrix_sel_data(X_matrix,lista_feature_names,lista_idx_filas=idx_test ,lista_feature_sel=lista_features)
print(X_test.shape)
X_val   = X_matrix_sel_data(X_matrix,lista_feature_names,lista_idx_filas=idx_val ,lista_feature_sel=lista_features)
print(X_val.shape)


(6572, 354)
(2226, 354)
(2173, 354)


In [54]:
modelo_1=lista_modelos[0]['model']
modelo_2=lista_modelos[4]['model']
modelo_3=lista_modelos[8]['model']

### <span style="color:#2ca02c">4.1.1. Ensemble soft-voting</span>

In [55]:

ensemble_voting = VotingClassifier(
    estimators=[
        ('rf', modelo_1),
        ('xgb', modelo_2),
        ('lgbm', modelo_3)
    ],
    voting='soft'  # Promedia las probabilidades de los tres modelos
)


ensemble_voting.fit(X_val, Y_val_encoded)



,"estimators estimators: list of (str, estimator) tuplesInvoking the ``fit`` method on the ``VotingClassifier`` will fit clonesof those original estimators that will be stored in the class attribute``self.estimators_``. An estimator can be set to ``'drop'`` using:meth:`set_params`... versionchanged:: 0.21 ``'drop'`` is accepted. Using None was deprecated in 0.22 and support was removed in 0.24.","[('rf', ...), ('xgb', ...), ...]"
,"voting voting: {'hard', 'soft'}, default='hard'If 'hard', uses predicted class labels for majority rule voting.Else if 'soft', predicts the class label based on the argmax ofthe sums of the predicted probabilities, which is recommended foran ensemble of well-calibrated classifiers.",'soft'
,"weights weights: array-like of shape (n_classifiers,), default=NoneSequence of weights (`float` or `int`) to weight the occurrences ofpredicted class labels (`hard` voting) or class probabilitiesbefore averaging (`soft` voting). Uses uniform weights if `None`.",None
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for ``fit``.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionadded:: 0.18",None
,"flatten_transform flatten_transform: bool, default=TrueAffects shape of transform output only when voting='soft'If voting='soft' and flatten_transform=True, transform method returnsmatrix with shape (n_samples, n_classifiers * n_classes). Ifflatten_transform=False, it returns(n_classifiers, n_samples, n_classes).",True
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting will be printed as itis completed... versionadded:: 0.23",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels.","ndarray[int64](53,)","[ 0, 1, 2,...,50,51,52]"
"estimators_ estimators_: list of classifiersThe collection of fitted sub-estimators as defined in ``estimators``that are not 'drop'. Note that sub-estimators are always fitted oninteger-encoded labels (see ``le_`` attribute). When ``y`` containsnon-integer class labels (e.g. strings), use ``le_.inverse_transform``to map predictions back to the original label space.",list,"[RandomForestC...dom_state=100), XGBClassifier...ree=None, ...), LGBMClassifie... verbosity=-1)]"
le_ le_: :class:`~sklearn.preprocessing.LabelEncoder`Transformer used to encode the labels during fit and decode duringprediction. Sub-estimators in ``estimators_`` are fitted on theinteger-encoded labels produced by this encoder.,LabelEncoder,LabelEncoder()
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying classifier exposes such an attribute when fit... versionadded:: 0.24,int,354


In [56]:
model_name = 'Ensemble Soft Voting(3 puntos todas las features) (metricas)'

paquete_ensemble = {}
paquete_ensemble['model_name']       = model_name
paquete_ensemble['model']            = ensemble_voting
paquete_ensemble['lista_features']   = lista_features
paquete_ensemble['datos']            = 'metricas_lags'
paquete_ensemble['algoritmo']        = 'Voting'


resultado, reporte_dict, Y_pred_probabilidades = evaluar_modelo(paquete_ensemble, lista_features, X_test, Y_test_etiquetas, encoder_etiquetas)

accuracy        = resultado['accuracy']
macro_f1        = resultado['f1']
macro_precision = resultado['precision']
macro_recall    = resultado['recall']
print(f"Model = {model_name:40s} Accuracy = {accuracy:.4f} | Macro F1 = {macro_f1:.4f} | Macro Precision = {macro_precision:.4f} | Macro Recall = {macro_recall:.4f} ")


Model = Ensemble Soft Voting(3 puntos todas las features) (metricas) Accuracy = 0.5517 | Macro F1 = 0.5470 | Macro Precision = 0.5500 | Macro Recall = 0.5517 


In [57]:
print(paquete_ensemble['classification_report_str'])


                                               precision    recall  f1-score   support

           access_token_auth_header_error_401       0.52      0.38      0.44        42
          access_token_authorization_form_401       0.41      0.33      0.37        42
         access_token_client_id_not_found_404       0.33      0.40      0.37        42
         access_token_client_secret_wrong_401       0.33      0.48      0.39        42
             access_token_form_urlencoded_400       0.53      0.40      0.46        42
          access_token_illegal_grant_type_400       0.45      0.43      0.44        42
access_token_missing_authorization_header_400       0.41      0.31      0.35        42
     authorization_code_client_id_missing_400       0.40      0.38      0.39        42
     authorization_code_invalid_client_id_404       0.33      0.31      0.32        42
      authorization_code_invalid_password_401       0.53      0.62      0.57        42
 authorization_code_missing_response_type_

In [58]:
paquete_ensemble['model_name']

'Ensemble Soft Voting(3 puntos todas las features) (metricas)'

In [59]:
joblib.dump(paquete_ensemble, f"{FOLDER_SALIDA}/models/{paquete_ensemble['model_name']}.joblib")

['result/models/Ensemble Soft Voting(3 puntos todas las features) (metricas).joblib']

In [95]:

ensemble_voting = VotingClassifier(
    estimators=[
        ('xgb', modelo_2),
        ('lgbm', modelo_3)
    ],
    voting='soft'  # Promedia las probabilidades de los tres modelos
)


ensemble_voting.fit(X_val, Y_val_encoded)


,"estimators estimators: list of (str, estimator) tuplesInvoking the ``fit`` method on the ``VotingClassifier`` will fit clonesof those original estimators that will be stored in the class attribute``self.estimators_``. An estimator can be set to ``'drop'`` using:meth:`set_params`... versionchanged:: 0.21 ``'drop'`` is accepted. Using None was deprecated in 0.22 and support was removed in 0.24.","[('xgb', ...), ('lgbm', ...)]"
,"voting voting: {'hard', 'soft'}, default='hard'If 'hard', uses predicted class labels for majority rule voting.Else if 'soft', predicts the class label based on the argmax ofthe sums of the predicted probabilities, which is recommended foran ensemble of well-calibrated classifiers.",'soft'
,"weights weights: array-like of shape (n_classifiers,), default=NoneSequence of weights (`float` or `int`) to weight the occurrences ofpredicted class labels (`hard` voting) or class probabilitiesbefore averaging (`soft` voting). Uses uniform weights if `None`.",None
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for ``fit``.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionadded:: 0.18",None
,"flatten_transform flatten_transform: bool, default=TrueAffects shape of transform output only when voting='soft'If voting='soft' and flatten_transform=True, transform method returnsmatrix with shape (n_samples, n_classifiers * n_classes). Ifflatten_transform=False, it returns(n_classifiers, n_samples, n_classes).",True
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting will be printed as itis completed... versionadded:: 0.23",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels.","ndarray[int64](53,)","[ 0, 1, 2,...,50,51,52]"
"estimators_ estimators_: list of classifiersThe collection of fitted sub-estimators as defined in ``estimators``that are not 'drop'. Note that sub-estimators are always fitted oninteger-encoded labels (see ``le_`` attribute). When ``y`` containsnon-integer class labels (e.g. strings), use ``le_.inverse_transform``to map predictions back to the original label space.",list,"[XGBClassifier...ree=None, ...), LGBMClassifie... verbosity=-1)]"
le_ le_: :class:`~sklearn.preprocessing.LabelEncoder`Transformer used to encode the labels during fit and decode duringprediction. Sub-estimators in ``estimators_`` are fitted on theinteger-encoded labels produced by this encoder.,LabelEncoder,LabelEncoder()
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying classifier exposes such an attribute when fit... versionadded:: 0.24,int,993


In [96]:
model_name = 'Ensemble Soft Voting(3 puntos todas las features) XGBOOST+lightGBM'

paquete_ensemble = {}
paquete_ensemble['model_name']       = model_name
paquete_ensemble['model']            = ensemble_voting
paquete_ensemble['lista_features']   = lista_features
paquete_ensemble['datos']            = 'metricas_log_lags'
paquete_ensemble['algoritmo']        = 'Voting'


resultado, reporte_dict, Y_pred_probabilidades = evaluar_modelo(paquete_ensemble, lista_features, X_test, Y_test_etiquetas, encoder_etiquetas)

accuracy        = resultado['accuracy']
macro_f1        = resultado['f1']
macro_precision = resultado['precision']
macro_recall    = resultado['recall']
print(f"Model = {model_name:40s} Accuracy = {accuracy:.4f} | Macro F1 = {macro_f1:.4f} | Macro Precision = {macro_precision:.4f} | Macro Recall = {macro_recall:.4f} ")


Model = Ensemble Soft Voting(3 puntos todas las features) XGBOOST+lightGBM Accuracy = 0.5954 | Macro F1 = 0.5878 | Macro Precision = 0.5857 | Macro Recall = 0.5954 


In [97]:
joblib.dump(paquete_ensemble, f"{FOLDER_SALIDA}/models/{paquete_ensemble['model_name']}.joblib")

['result/models/Ensemble Soft Voting(3 puntos todas las features) XGBOOST+lightGBM.joblib']

### <span style="color:#2ca02c">4.1.2. Ensemble hard-voting</span>

In [60]:

ensemble_voting = VotingClassifier(
    estimators=[
        ('rf', modelo_1),
        ('xgb', modelo_2),
        ('lgbm', modelo_3)
    ],
    voting='hard'  # Promedia las probabilidades de los tres modelos
)


ensemble_voting.fit(X_val, Y_val_encoded)


,"estimators estimators: list of (str, estimator) tuplesInvoking the ``fit`` method on the ``VotingClassifier`` will fit clonesof those original estimators that will be stored in the class attribute``self.estimators_``. An estimator can be set to ``'drop'`` using:meth:`set_params`... versionchanged:: 0.21 ``'drop'`` is accepted. Using None was deprecated in 0.22 and support was removed in 0.24.","[('rf', ...), ('xgb', ...), ...]"
,"voting voting: {'hard', 'soft'}, default='hard'If 'hard', uses predicted class labels for majority rule voting.Else if 'soft', predicts the class label based on the argmax ofthe sums of the predicted probabilities, which is recommended foran ensemble of well-calibrated classifiers.",'hard'
,"weights weights: array-like of shape (n_classifiers,), default=NoneSequence of weights (`float` or `int`) to weight the occurrences ofpredicted class labels (`hard` voting) or class probabilitiesbefore averaging (`soft` voting). Uses uniform weights if `None`.",None
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for ``fit``.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionadded:: 0.18",None
,"flatten_transform flatten_transform: bool, default=TrueAffects shape of transform output only when voting='soft'If voting='soft' and flatten_transform=True, transform method returnsmatrix with shape (n_samples, n_classifiers * n_classes). Ifflatten_transform=False, it returns(n_classifiers, n_samples, n_classes).",True
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting will be printed as itis completed... versionadded:: 0.23",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels.","ndarray[int64](53,)","[ 0, 1, 2,...,50,51,52]"
"estimators_ estimators_: list of classifiersThe collection of fitted sub-estimators as defined in ``estimators``that are not 'drop'. Note that sub-estimators are always fitted oninteger-encoded labels (see ``le_`` attribute). When ``y`` containsnon-integer class labels (e.g. strings), use ``le_.inverse_transform``to map predictions back to the original label space.",list,"[RandomForestC...dom_state=100), XGBClassifier...ree=None, ...), LGBMClassifie... verbosity=-1)]"
le_ le_: :class:`~sklearn.preprocessing.LabelEncoder`Transformer used to encode the labels during fit and decode duringprediction. Sub-estimators in ``estimators_`` are fitted on theinteger-encoded labels produced by this encoder.,LabelEncoder,LabelEncoder()
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying classifier exposes such an attribute when fit... versionadded:: 0.24,int,354


In [61]:
model_name = 'Ensemble Hard Voting(3 puntos todas las features)(metricas)'

paquete_ensemble = {}
paquete_ensemble['model_name']       = model_name
paquete_ensemble['model']            = ensemble_voting
paquete_ensemble['lista_features']   = lista_features
paquete_ensemble['datos']            = 'metricas_lags'
paquete_ensemble['algoritmo']        = 'Voting'


resultado, reporte_dict, Y_pred_probabilidades = evaluar_modelo(paquete_ensemble, lista_features, X_test, Y_test_etiquetas, encoder_etiquetas)

accuracy        = resultado['accuracy']
macro_f1        = resultado['f1']
macro_precision = resultado['precision']
macro_recall    = resultado['recall']
print(f"Model = {model_name:40s} Accuracy = {accuracy:.4f} | Macro F1 = {macro_f1:.4f} | Macro Precision = {macro_precision:.4f} | Macro Recall = {macro_recall:.4f} ")


Model = Ensemble Hard Voting(3 puntos todas las features)(metricas) Accuracy = 0.5427 | Macro F1 = 0.5382 | Macro Precision = 0.5427 | Macro Recall = 0.5427 


In [62]:
joblib.dump(paquete_ensemble, f"{FOLDER_SALIDA}/models/{paquete_ensemble['model_name']}.joblib")

['result/models/Ensemble Hard Voting(3 puntos todas las features)(metricas).joblib']

In [64]:
joblib.dump(paquete_ensemble, f"{FOLDER_SALIDA}/models/{paquete_ensemble['model_name']}.joblib")

['result/models/Ensemble Hard Voting(3 puntos todas las features)(metricas).joblib']

### <span style="color:#2ca02c">4.1.3. Ensemble Stacking usando probabilidades que generan los modelos originales

In [65]:
pred_m1 = modelo_1.predict_proba(X_val)
pred_m2 = modelo_2.predict_proba(X_val)
pred_m3 = modelo_3.predict_proba(X_val)

In [66]:
X_meta_train = np.hstack([pred_m1, pred_m2, pred_m3])

In [67]:
from sklearn.linear_model import LogisticRegression

# Creamos y entrenamos el meta-modelo
meta_modelo = LogisticRegression(max_iter=1000)
meta_modelo.fit(X_meta_train, Y_val_encoded)

,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is '

In [68]:
test_pred_m1 = modelo_1.predict_proba(X_test)
test_pred_m2 = modelo_2.predict_proba(X_test)
test_pred_m3 = modelo_3.predict_proba(X_test)

X_meta_test = np.hstack([test_pred_m1, test_pred_m2, test_pred_m3])

# 2. El meta-modelo predice las clases finales
Y_pred_stack       = meta_modelo.predict(X_meta_test)
Y_pred_proba_stack = meta_modelo.predict_proba(X_meta_test)
Y_pred_tipos       = encoder_etiquetas.inverse_transform(Y_pred_stack)

In [69]:



print("=== INFORME DE CLASIFICACIÓN (STACKING) ===")
classification_report_str =classification_report(Y_test_etiquetas, Y_pred_tipos)

accuracy               = accuracy_score(Y_test_etiquetas, Y_pred_tipos)
macro_f1               = f1_score(Y_test_etiquetas, Y_pred_tipos, average='macro')
macro_precision        = precision_score(Y_test_etiquetas, Y_pred_tipos, average='macro', zero_division=0)
macro_recall           = recall_score(Y_test_etiquetas, Y_pred_tipos, average='macro', zero_division=0)
reporte_dict           = classification_report(Y_test_etiquetas, Y_pred_tipos, output_dict=True, zero_division=0)
Y_pred_probabilidades  = meta_modelo.predict_proba(X_meta_test)

print(f"Resultados Globales -> Accuracy: {accuracy:.4f} | Macro F1: {macro_f1:.4f} | Macro Precision: {macro_precision:.4f} | Macro Recall: {macro_recall:.4f}")

=== INFORME DE CLASIFICACIÓN (STACKING) ===
Resultados Globales -> Accuracy: 0.7848 | Macro F1: 0.7847 | Macro Precision: 0.7875 | Macro Recall: 0.7848


In [73]:
paquete_ensemble={}
nombre_algoritmo                              = 'Meta-modelo de regresion logistica'
model_name                                    = f'{nombre_algoritmo} (RF+XGBOOST+lightGBM) (metricas)'
resultado                                     = {'accuracy': accuracy,  'precision':macro_precision   , 'recall':macro_recall  , 'f1':macro_f1 }
paquete_ensemble['model_name']                = model_name
paquete_ensemble['model']                     = meta_modelo
paquete_ensemble['resultado']                 = resultado
paquete_ensemble['probabilidades']            = Y_pred_probabilidades
paquete_ensemble['reporte_dict']              = reporte_dict
paquete_ensemble['classification_report_str'] = classification_report_str
paquete_ensemble['parametros']                = None
paquete_ensemble['lista_features']            = lista_features_names_3puntos
paquete_ensemble['N_features']                = len(paquete_ensemble['lista_features'])
paquete_ensemble['N_features_unicas']         = len(get_lista_feature_sin_lag(paquete_ensemble['lista_features']))
paquete_ensemble['datos']                     = 'metricas_lags'
paquete_ensemble['algoritmo']                 = nombre_algoritmo
paquete_ensemble['lista_modelos_orig']        = [ 
    'result/models/RandomForest (metricas_logs 3 puntos)(todas las features).joblib',
    'result/models/XGBoost (metricas_logs 3 puntos)(todas las features).joblib',
    'result/models/LightGBM (metricas_logs 3 puntos)(todas las features).joblib'
]



In [74]:
joblib.dump(paquete_ensemble, f"{FOLDER_SALIDA}/models/{paquete_ensemble['model_name']}.joblib")


['result/models/Meta-modelo de regresion logistica (RF+XGBOOST+lightGBM) (metricas).joblib']

In [ ]:
del paquete_ensemble
gc.collect()

In [127]:
print('Comparacion de modelos originales con todas las metricas_log y ensembles')
print('-'*80)
tabla_comparacion_modelos([
    'result/models/RandomForest (metricas_logs 3 puntos)(todas las features).joblib',
    'result/models/XGBoost (metricas_logs 3 puntos)(todas las features).joblib',
    'result/models/LightGBM (metricas_logs 3 puntos)(todas las features).joblib',
    'result/models/Ensemble Soft Voting(3 puntos todas las features) XGBOOST+lightGBM.joblib',
    'result/models/Ensemble Hard Voting(3 puntos todas las features)  XGBOOST+lightGBM.joblib',
    'result/models/Meta-modelo de regresion logistica (RF+XGBOOST+lightGBM).joblib'
]
)

Comparacion de modelos originales con todas las metricas_log y ensembles
--------------------------------------------------------------------------------
Model = RandomForest (metricas_logs 3 puntos)(todas las features)              Accuracy = 0.6232 | Macro F1 = 0.6255 | Macro Precision = 0.6232 | Macro Recall = 0.6192
Model = XGBoost (metricas_logs 3 puntos)(todas las features)                   Accuracy = 0.6882 | Macro F1 = 0.6911 | Macro Precision = 0.6882 | Macro Recall = 0.6878
Model = LightGBM (metricas_logs 3 puntos)(todas las features)                  Accuracy = 0.6940 | Macro F1 = 0.6938 | Macro Precision = 0.6940 | Macro Recall = 0.6924
Model = Ensemble Soft Voting(3 puntos todas las features) XGBOOST+lightGBM     Accuracy = 0.5954 | Macro F1 = 0.5857 | Macro Precision = 0.5954 | Macro Recall = 0.5878
Model = Ensemble Hard Voting(3 puntos todas las features)  XGBOOST+lightGBM    Accuracy = 0.5974 | Macro F1 = 0.5902 | Macro Precision = 0.5974 | Macro Recall = 0.5905
Model 

## <span style="color:#2ca02c">4.2. Ensemble de los mejores modelos con features reducidas</span>

In [227]:
lista_filenames_ensemble=[
    'result/models/RandomForest (metricas_logs 3 puntos)   N_feat=63.joblib',
    'result/models/XGBoost (metricas_logs 3 puntos)   N_feat=102.joblib',
    'result/models/LightGBM (metricas_logs 3 puntos)   N_feat=147.joblib'
]
    

In [228]:
lista_paquetes = []
for filename in lista_filenames_ensemble:
    paquete_modelo = joblib.load(filename)
    lista_paquetes.append(paquete_modelo)

    

In [229]:
lista_pred_m = []

for paquete in lista_paquetes:
    lista_features=paquete['lista_features']
    X_val_sub = X_matrix_sel_data(X_matrix, lista_feature_names, lista_idx_filas=idx_val, lista_feature_sel=lista_features)
    pred_m = paquete['model'].predict_proba(X_val_sub)
    lista_pred_m.append(pred_m)
    



In [230]:
X_meta_train = np.hstack(lista_pred_m)
meta_modelo = LogisticRegression(max_iter=1000)
meta_modelo.fit(X_meta_train, Y_val_encoded)

,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is '

In [231]:
lista_test_pred_m = []

for paquete in lista_paquetes:
    lista_features=paquete['lista_features']
    X_test_sub = X_matrix_sel_data(X_matrix, lista_feature_names, lista_idx_filas=idx_test, lista_feature_sel=lista_features)
    test_pred_m = paquete['model'].predict_proba(X_test_sub)
    lista_test_pred_m.append(test_pred_m)




In [232]:
lista_test_pred_m[2].shape

(3445, 53)

In [233]:
X_meta_test = np.hstack(lista_test_pred_m)
Y_pred_stack       = meta_modelo.predict(X_meta_test)
Y_pred_proba_stack = meta_modelo.predict_proba(X_meta_test)
Y_pred_tipos       = encoder_etiquetas.inverse_transform(Y_pred_stack)


In [234]:

print("=== INFORME DE CLASIFICACIÓN (STACKING) ===")
classification_report_str = classification_report(Y_test_etiquetas, Y_pred_tipos)
reporte_dict              = classification_report(Y_test_etiquetas, Y_pred_tipos, output_dict=True, zero_division=0)


accuracy                  = accuracy_score(Y_test_etiquetas, Y_pred_tipos)
macro_f1                  = f1_score(Y_test_etiquetas, Y_pred_tipos, average='macro')
macro_precision           = precision_score(Y_test_etiquetas, Y_pred_tipos, average='macro', zero_division=0)
macro_recall              = recall_score(Y_test_etiquetas, Y_pred_tipos, average='macro', zero_division=0)
reporte_dict              = classification_report(Y_test_etiquetas, Y_pred_tipos, output_dict=True, zero_division=0)
Y_pred_probabilidades     = meta_modelo.predict_proba(X_meta_test)

print(f"Resultados Globales -> Accuracy: {accuracy:.4f} | Macro F1: {macro_f1:.4f} | Macro Precision: {macro_precision:.4f} | Macro Recall: {macro_recall:.4f}")

=== INFORME DE CLASIFICACIÓN (STACKING) ===
Resultados Globales -> Accuracy: 0.7013 | Macro F1: 0.7032 | Macro Precision: 0.7106 | Macro Recall: 0.7013


In [235]:
paquete_ensemble={}
nombre_algoritmo                              = 'Meta-modelo de regresion logistica'
model_name                                    = f'{nombre_algoritmo} mejores modelos (RF+XGBOOST+lightGBM) '
resultado                                     = {'accuracy': accuracy,  'precision':macro_precision   , 'recall':macro_recall  , 'f1':macro_f1 }
paquete_ensemble['model_name']                = model_name
paquete_ensemble['model']                     = meta_modelo
paquete_ensemble['resultado']                 = resultado
paquete_ensemble['probabilidades']            = Y_pred_probabilidades
paquete_ensemble['reporte_dict']              = reporte_dict
paquete_ensemble['classification_report_str'] = classification_report_str
paquete_ensemble['parametros']                = None
paquete_ensemble['lista_features']            = lista_features_names_3puntos
paquete_ensemble['N_features']                = len(paquete_ensemble['lista_features'])
paquete_ensemble['N_features_unicas']         = len(get_lista_feature_sin_lag(paquete_ensemble['lista_features']))
paquete_ensemble['datos']                     = 'metricas_log_lags'
paquete_ensemble['algoritmo']                 = nombre_algoritmo
paquete_ensemble['lista_modelos_orig']        = lista_filenames_ensemble


In [236]:
joblib.dump(paquete_ensemble, f"{FOLDER_SALIDA}/models/{paquete_ensemble['model_name']}.joblib")


['result/models/Meta-modelo de regresion logistica mejores modelos (RF+XGBOOST+lightGBM) .joblib']

In [237]:
lista_filenames_ensemble.append( f"{FOLDER_SALIDA}/models/{paquete_ensemble['model_name']}.joblib")

In [238]:
print('Comparacion de modelos originales con todas las metricas_log y ensembles')
print('-'*80)
tabla_comparacion_modelos(lista_filenames_ensemble)

Comparacion de modelos originales con todas las metricas_log y ensembles
--------------------------------------------------------------------------------
Model = RandomForest (metricas_logs 3 puntos)   N_feat=63                      Accuracy = 0.6374 | Macro F1 = 0.6414 | Macro Precision = 0.6374 | Macro Recall = 0.6352
Model = XGBoost (metricas_logs 3 puntos)   N_feat=102                          Accuracy = 0.6978 | Macro F1 = 0.6999 | Macro Precision = 0.6978 | Macro Recall = 0.6969
Model = LightGBM (metricas_logs 3 puntos)   N_feat=147                         Accuracy = 0.7022 | Macro F1 = 0.7040 | Macro Precision = 0.7022 | Macro Recall = 0.7011
Model = Meta-modelo de regresion logistica mejores modelos (RF+XGBOOST+lightGBM)  Accuracy = 0.7013 | Macro F1 = 0.7106 | Macro Precision = 0.7013 | Macro Recall = 0.7032


In [ ]:
del paquete_ensemble,lista_paquetes,X_meta_train,meta_modelo,lista_test_pred_m,X_meta_test,Y_pred_stack,Y_pred_proba_stack,Y_pred_tipos,paquete_ensemble
gc.collect()

# <span style="color:#2ca02c">5. Revision complementariedad de modelos</span>

In [ ]:
def calcular_potencial_mejora(y_true, y_pred_1, y_pred_2, average='macro'):
    """
    Calcula un valor numérico que indica si un modelo híbrido mejorará 
    el rendimiento de los modelos individuales.
    """
    y_true = np.array(y_true)
    y_pred_1 = np.array(y_pred_1)
    y_pred_2 = np.array(y_pred_2)
    
    # 1. Métricas base de los modelos individuales (ej. Macro F1)
    score_1 = f1_score(y_true, y_pred_1, average=average, zero_division=0)
    score_2 = f1_score(y_true, y_pred_2, average=average, zero_division=0)
    mejor_base = max(score_1, score_2)
    
    # 2. Techo Teórico (Oracle): Simulación perfecta donde elegimos el acierto 
    # de cualquiera de los dos modelos si al menos uno acierta la muestra.
    aciertos_1 = (y_pred_1 == y_true)
    aciertos_2 = (y_pred_2 == y_true)
    acierto_or = aciertos_1 | aciertos_2
    
    # Si simulamos que un hipotético híbrido perfecto rescata todos los aciertos posibles:
    # (Para muestras donde ambos fallan, mantenemos una predicción al azar o del mejor)
    # Una aproximación numérica directa del límite superior (Techo Oracle):
    techo_teorico = np.mean(acierto_or) # (Si usamos accuracy como base de techo)
    
    # 3. CÁLCULO DEL NÚMERO CLAVE: "Margen de Mejora Potencial" (Delta)
    # Mide cuántos puntos porcentuales hay de margen entre el mejor modelo y el límite teórico.
    # (Lo calculamos con el acierto binario general o adaptado al F1)
    acc_1 = accuracy_score(y_true, y_pred_1)
    acc_2 = accuracy_score(y_true, y_pred_2)
    mejor_acc_base = max(acc_1, acc_2)
    
    delta_mejora = techo_teorico - mejor_acc_base
    
    # 4. Tasa de Desacuerdo (Diversidad)
    tasa_desacuerdo = np.mean(aciertos_1 != aciertos_2)
    
    print("=" * 60)
    print("      INDICADOR NUMÉRICO DE POTENCIAL DE MEJORA")
    print("=" * 60)
    print(f"Score (Macro F1) Modelo 1:    {score_1:.4f}")
    print(f"Score (Macro F1) Modelo 2:    {score_2:.4f}")
    print(f"Mejor Modelo Individual:      {mejor_base:.4f}")
    print("-" * 60)
    print(f"Tasa de Desacuerdo (Diversidad): {tasa_desacuerdo * 100:.2f}%")
    print(f"Margen de Mejora Teórico (Delta): +{delta_mejora * 100:.2f}%  <-- NÚMERO CLAVE")
    print("=" * 60)
    
    # Interpretación numérica estricta:
    if delta_mejora > 0.03:  # Más de un 3% de margen teórico
        print("🟢 Veredicto: ALTO POTENCIAL DE MEJORA (> 3% de margen).")
    elif delta_mejora > 0.01: # Entre 1% y 3%
        print("🟡 Veredicto: MEJORA MODERADA O MARGINAL (1% - 3%).")
    else:
        print("🔴 Veredicto: SIN POTENCIAL DE MEJORA (< 1%). Los modelos son redundantes.")
        
    return {
        'mejor_base': mejor_base,
        'techo_teorico': techo_teorico,
        'delta_mejora': delta_mejora,
        'tasa_desacuerdo': tasa_desacuerdo
    }

In [95]:
lista_filenames=[
    'result/models/RandomForest (metricas_logs 3 puntos)(todas las features).joblib',
    'result/models/RandomForest (metricas_logs 3 puntos)   N_feat=63.joblib',
    
    'result/models/RandomForest (metricas_logs 1 punto suma)(todas las features).joblib',
    'result/models/RandomForest (metricas_logs 1 punto suma)   N_feat=78.joblib',
    
    'result/models/XGBoost (metricas_logs 3 puntos)(todas las features).joblib',
    'result/models/XGBoost (metricas_logs 3 puntos)   N_feat=102.joblib',
    
    'result/models/XGBoost (metricas_logs 1 punto suma)(todas las features).joblib',
    'result/models/XGBoost (metricas_logs 1 punto suma)   N_feat=160.joblib',
    
    'result/models/LightGBM (metricas_logs 3 puntos)(todas las features).joblib',
    'result/models/LightGBM (metricas_logs 3 puntos)   N_feat=147.joblib',
    
    'result/models/LightGBM (metricas_logs 1 punto suma)(todas las features).joblib',
    'result/models/LightGBM (metricas_logs 1 punto suma)   N_feat=202.joblib',
    
#     'result/models/RandomForest (metricas_log) N_feat=44.joblib',
#     'result/models/XGBoost con metricas_logs (3 puntos y todas las metricas).joblib',
#     'result/models/XGBoost (metricas_log) N_feat=93.joblib',
#    'result/models/XGBoost con metricas_logs (1 punto 3 intervalos todas las metricas).joblib'
#     'result/models/XGBoost (metricas_log) N_feat=110.joblib'
]

In [96]:
paquete_modelo['resultado']

{'accuracy': 0.6882438316400581,
 'precision': 0.6910796659371462,
 'recall': 0.6882438316400581,
 'f1': 0.6878356873422209}

In [97]:
lista_modelos=[]
for filename in lista_filenames:
    paquete_modelo = joblib.load(filename)
    lista_modelos.append(paquete_modelo)
    


In [98]:
for paquete_modelo in lista_modelos:
    model_name      = paquete_modelo['model_name']
    accuracy        = paquete_modelo['resultado']['accuracy']
    macro_f1        = paquete_modelo['resultado']['precision']
    macro_precision = paquete_modelo['resultado']['recall']
    macro_recall    = paquete_modelo['resultado']['f1']
    print(f"Model = {model_name:70s} Accuracy = {accuracy:.4f} | Macro F1 = {macro_f1:.4f} | Macro Precision = {macro_precision:.4f} | Macro Recall = {macro_recall:.4f}")


Model = RandomForest (metricas_logs 3 puntos)(todas las features)              Accuracy = 0.6232 | Macro F1 = 0.6255 | Macro Precision = 0.6232 | Macro Recall = 0.6192
Model = RandomForest (metricas_logs 3 puntos)   N_feat=63                      Accuracy = 0.6374 | Macro F1 = 0.6414 | Macro Precision = 0.6374 | Macro Recall = 0.6352
Model = RandomForest (metricas_logs 1 punto suma)(todas las features)          Accuracy = 0.6136 | Macro F1 = 0.6076 | Macro Precision = 0.6136 | Macro Recall = 0.6008


KeyError: 'precision'

In [115]:
paquete_modelo['resultado']

{'accuracy': 0.6275761973875181,
 'macro_f1': 0.6197418658287879,
 'macro_precision': 0.6283339292003264,
 'macro_recall': 0.6275761973875182}

In [ ]:
lista_Y_pred=[]
Y_pred =  predict_modelo(lista_modelos[0],lista_feature_names1_sum,X_test_sum)
lista_Y_pred.append(Y_pred)
Y_pred =  predict_modelo(lista_modelos[1],lista_feature_names1,X_test)
lista_Y_pred.append(Y_pred)
# Y_pred_tipos                                = encoder_etiquetas.inverse_transform(Y_pred)

In [ ]:
calcular_potencial_mejora(Y_test_encoded,lista_Y_pred[0],lista_Y_pred[1])

In [ ]:

from sklearn.linear_model import LogisticRegression

# 1. Obtener las matrices de validación con las features específicas de cada modelo
X_val_sub_1 = X_matrix_sel_data(X_matrix, lista_feature_names, lista_idx_filas=idx_val, lista_feature_sel=lista_modelos[0]['lista_features'])
X_val_sub_2 = X_matrix_sel_data(X_matrix, lista_feature_names, lista_idx_filas=idx_val, lista_feature_sel=lista_modelos[1]['lista_features'])

# 2. Generar las probabilidades sobre ESE conjunto de validación (datos no vistos)
probs_val_1 = lista_modelos[0]['model'].predict_proba(X_val_sub_1)
probs_val_2 = lista_modelos[1]['model'].predict_proba(X_val_sub_2)

# 3. Construir las meta-features de entrenamiento para el metamodelo
X_meta_train = np.hstack((probs_val_1, probs_val_2))

# 4. Entrenar el metamodelo
meta_model = LogisticRegression(max_iter=1000,class_weight='balanced')
meta_model.fit(X_meta_train, Y_val_encoded)


X_test_sub_1 = X_matrix_sel_data(X_matrix, lista_feature_names, lista_idx_filas=idx_test, lista_feature_sel=lista_modelos[0]['lista_features'])
X_test_sub_2 = X_matrix_sel_data(X_matrix, lista_feature_names, lista_idx_filas=idx_test, lista_feature_sel=lista_modelos[1]['lista_features'])

probs_test_1 = lista_modelos[0]['model'].predict_proba(X_test_sub_1)
probs_test_2 = lista_modelos[1]['model'].predict_proba(X_test_sub_2)

# 5. Para test (haces lo mismo pero con X_test)
X_meta_test = np.hstack((probs_test_1, probs_test_2))
y_pred_meta = meta_model.predict(X_meta_test)

In [ ]:
Y_pred_tipos                                = encoder_etiquetas.inverse_transform(y_pred_meta)

In [ ]:
print(classification_report(Y_test_etiquetas, Y_pred_tipos))